# Batching Inputs Together

## Sentences we want to group inside a batch will often have different lengths

In [2]:
from transformers import AutoTokenizer

checkpoint = "Qwen/Qwen2.5-1.5B"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
sentences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this.",
]
tokens = [tokenizer.tokenize(sentence) for sentence in sentences]
ids = [tokenizer.convert_tokens_to_ids(token) for token in tokens]
print(ids[0])
print(ids[1])

[40, 3003, 1012, 8580, 369, 264, 472, 35268, 16281, 3308, 847, 4361, 2272, 13]
[40, 12213, 419, 13]


## You can't build a tensor with lists of different lengths

In [4]:
import torch

ids = [[40, 3003, 1012, 8580, 369, 264, 472, 35268, 16281, 3308, 847, 4361, 2272, 13],
       [40, 12213, 419, 13]]

input_ids = torch.tensor(ids)

ValueError: expected sequence of length 14 at dim 1 (got 4)

## Which is why we usually pad the smaller sentences to the length of the longest one!

In [5]:
import torch

ids = [[40, 3003, 1012, 8580, 369, 264, 472, 35268, 16281, 3308, 847, 4361, 2272, 13],
       [40,12213,  419,   13,   0,   0,   0,     0,     0,    0,   0,    0,    0,  0]]

input_ids = torch.tensor(ids)

- you can also truncate the longer sentence however it would cause information loss
- truncating is only done when the sentence is longer than what the model can handle

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
tokenizer.pad_token_id

151643

- The model is pretrained with a padding id which can be found when running this code
- **Do not choose a random padding id use the pretrained one**

## But just passing this through a Transformers model will not give the right results

In [7]:
from transformers import AutoModelForSequenceClassification

ids1 = torch.tensor(
    [[40, 3003, 1012, 8580, 369, 264, 472, 35268, 16281, 3308, 847, 4361, 2272, 13]]
)
ids2 = torch.tensor([[40, 12213, 419, 13]])
all_ids = torch.tensor(
    [[40, 3003, 1012, 8580, 369, 264, 472, 35268, 16281, 3308, 847, 4361, 2272, 13],
       [40,12213,  419,   13,   151643,   151643,   151643,     151643,     151643,    151643,   151643,    151643,    151643,  151643]]
)

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
print(model(ids1).logits)
print(model(ids2).logits)
print(model(all_ids).logits)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 7021.95it/s]
[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tensor([[        nan, -9.9303e+34]], dtype=torch.bfloat16,
       grad_fn=<IndexBackward0>)
tensor([[        nan, -1.7524e+35]], dtype=torch.bfloat16,
       grad_fn=<IndexBackward0>)


ValueError: Cannot handle batch sizes > 1 if no padding token is defined.

- The padded sentence doesn't give the same results as the unpadded ids
- The transformers library requires an attention layer

## This is because the attention layers use the padding tokens in the context they look at for each token in the sentence

- To get the same results with or without padding we need to create an attention mask with 0 and 1s to tell the transformer what to ignore.

## To tell the attention layers to ignore the padding tokens, we need to pass them an attention mask.

In [11]:
model.config.pad_token_id = tokenizer.pad_token_id
all_ids = torch.tensor(
    [[40, 3003, 1012, 8580,      369,      264,      472,      35268,      16281,      3308,      847,      4361,      2272,      13],
     [40,12213,  419,   13,   151643,   151643,   151643,     151643,     151643,    151643,   151643,    151643,    151643,  151643]]
)
attention_mask = torch.tensor(
    [[ 1,    1,    1,    1,        1,        1,        1,          1,          1,         1,         1,         1,         1,       1],
     [ 1,    1,    1,    1,        0,        0,        0,          0,          0,         0,         0,         0,         0,       0]]
)

- Attention layers will ignore the tokens marked with 0 in the attention mask

## With the proper attention mask, predictions are the same for a given sentence, with or without padding.

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
output1 = model(ids1)
output2 = model(ids2)
print(output1.logits)
print(output2.logits)

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 6657.34it/s]
[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tensor([[       nan, 6.6877e+36]], dtype=torch.bfloat16,
       grad_fn=<IndexBackward0>)
tensor([[       nan, 7.1031e+36]], dtype=torch.bfloat16,
       grad_fn=<IndexBackward0>)


In [12]:
output = model(all_ids, attention_mask = attention_mask)
print(output.logits)

tensor([[       nan, 6.6877e+36],
        [       nan, 7.1031e+36]], dtype=torch.bfloat16,
       grad_fn=<IndexBackward0>)


## Used with padding=True, the tokenizer can directly prepare the inputs with padding and the proper attention mask

In [13]:
from transformers import AutoTokenizer

checkpoint = "Qwen/Qwen2.5-1.5B"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
sentences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this.",
]
print(tokenizer(sentences, padding=True))

{'input_ids': [[40, 3003, 1012, 8580, 369, 264, 472, 35268, 16281, 3308, 847, 4361, 2272, 13], [40, 12213, 419, 13, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}


- This is all done automatically by the tokenizer when you apply the flag padding = True
- It automatically creates the attention mask as well

# Hugging Face Datasets Overview

## The Datasets library allows you to easily download and cache datasets.

In [32]:
from datasets import load_dataset

gsm8k = load_dataset("openai/gsm8k", "main")
gpqa = load_dataset("Idavidrein/gpqa", "gpqa_main")

print("This is gsm8k \n")
print(gsm8k)
print("This is gpqa \n")
print(gpqa)

This is gsm8k 

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})
This is gpqa 

DatasetDict({
    train: Dataset({
        features: ['Pre-Revision Question', 'Pre-Revision Correct Answer', 'Pre-Revision Incorrect Answer 1', 'Pre-Revision Incorrect Answer 2', 'Pre-Revision Incorrect Answer 3', 'Pre-Revision Explanation', 'Self-reported question-writing time (minutes)', 'Question', 'Correct Answer', 'Incorrect Answer 1', 'Incorrect Answer 2', 'Incorrect Answer 3', 'Explanation', 'Revision Comments (from Question Writer)', 'Subdomain', "Writer's Difficulty Estimate", 'Extra Revised Question', 'Extra Revised Explanation', 'Extra Revised Correct Answer', 'Extra Revised Incorrect Answer 1', 'Extra Revised Incorrect Answer 2', 'Extra Revised Incorrect Answer 3', 'Non-Expert Validator Accuracy', 'Majority Non-Expert Vals Incorrect', 'Expert V

- Returns a DatasetDict which is a type of dictionary of the dataset

## We can access any split of the Dataset by its Key, then any element by index

In [21]:
gsm8k["train"]

Dataset({
    features: ['question', 'answer'],
    num_rows: 7473
})

In [22]:
gsm8k["train"][6]

{'question': 'Albert is wondering how much pizza he can eat in one day. He buys 2 large pizzas and 2 small pizzas. A large pizza has 16 slices and a small pizza has 8 slices. If he eats it all, how many pieces does he eat that day?',
 'answer': 'He eats 32 from the largest pizzas because 2 x 16 = <<2*16=32>>32\nHe eats 16 from the small pizza because 2 x 8 = <<2*8=16>>16\nHe eats 48 pieces because 32 + 16 = <<32+16=48>>48\n#### 48'}

- Everything is saved to disk using Apache Arrow so you wont run out of RAM

## You can also get a slice of the dataset

In [23]:
gsm8k['train'][:5]

{'question': ['Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
  'Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?',
  'Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?',
  'Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read?',
  'James writes a 3-page letter to 2 different friends twice a week.  How many pages does he write a year?'],
 'answer': ['Natalia sold 48/2 = <<48/2=24>>24 clips in May.\nNatalia sold 48+24 = <<48+24=72>>72 clips alt

## The features attributes gives us more information about each column

In [24]:
gsm8k["train"].features

{'question': Value('string'), 'answer': Value('string')}

## The map method allows you to apply a function over all the splits of a given dataset

In [30]:
from transformers import AutoTokenizer
checkpoint = "Qwen/Qwen2.5-1.5B"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(
        example["question"], example["answer"], padding="max_length", truncation=True, max_length=128
    )

tokenized_gsm8k = gsm8k.map(tokenize_function)
print(tokenized_gsm8k.column_names)

{'train': ['question', 'answer', 'input_ids', 'attention_mask'], 'test': ['question', 'answer', 'input_ids', 'attention_mask']}


## You can preprocess faster by using the option batched=True. The applied function will then receive multiple examples at each call

In [41]:
from transformers import AutoTokenizer

checkpoint = "Qwen/Qwen2.5-1.5B"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(examples):
    return tokenizer(
        examples["question"], examples["answer"], padding="max_length", truncation=True, max_length=128
    )
tokenized_gsm8k = gsm8k.map(tokenize_function, batched=True)
print(tokenized_gsm8k.column_names)

{'train': ['question', 'answer', 'input_ids', 'attention_mask'], 'test': ['question', 'answer', 'input_ids', 'attention_mask']}


## With just a few last tweaks, your dataset is then ready for training!

In [42]:
tokenized_gsm8k = tokenized_gsm8k.remove_columns(['question','answer'])
tokenized_gsm8k = tokenized_gsm8k.with_format("torch")
tokenized_gsm8k["train"]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 7473
})

In [43]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B")
print(tokenizer.chat_template)


{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n' }}
    {%- endif %}
{%- endif %}